In [2]:
# Run once if you see missing `matplotlib.backends.backend_agg` or `matplotlib_inline`.
# `%pip` installs into **this notebook kernel** (the `fk` env), unlike `!pip`.
%pip install --force-reinstall --no-cache-dir "matplotlib>=3.7" matplotlib-inline

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 47.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 36.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 43.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.8/16.8 MB 32.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 39.2 MB/s  0:00:00
  Attempting uninstall: traitlets
    Found existing installation: traitlets 5.14.3
    Uninstalling traitlets-5.14.3:
      Successfully uninstalled traitlets-5.14.3
  Attempting uninstall: six
    Found existing installation: six 1.17.0
    Uninstalling six-1.17.0:
      Successfully uninstalled six-1.17.0
  Attempting uninstall: pillow━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/13 [six]
    Found existing installation: pillow 12.2.0━━━━━━━━━━━━━━━━  1/13 [six]
    Uninstalling pillow-12.2.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  1/13 [six]
      Successfully uninstalled pillow-12.2.0━━━━━━━━━━━━━━━━━━  1/13 [six]
  

In [5]:
import os
# Do not set CUDA_VISIBLE_DEVICES to a single GPU if you want a *second* physical GPU:
# with only one visible device, PyTorch exposes it only as cuda:0.

import json
from PIL import Image
import matplotlib.pyplot as plt
import argparse
from datetime import datetime
import numpy as np
import sys
from pathlib import Path

# Notebook cwd is often repo root; launch_eval_runs.py and fks_utils live in text_to_image/
_cwd = Path.cwd().resolve()
_ti = _cwd / "text_to_image"
if _ti.is_dir():
    sys.path.insert(0, str(_ti))
sys.path.insert(0, str(_cwd))

import torch

_DESIRED_GPU_INDEX = 1  # physical index you want; must be < num GPUs *this process* sees
if torch.cuda.is_available():
    _n = torch.cuda.device_count()
    if _DESIRED_GPU_INDEX >= _n:
        print(
            f"Warning: wanted cuda:{_DESIRED_GPU_INDEX} but only {_n} GPU(s) visible "
            f"(valid: 0..{_n-1}). Using cuda:0. Check CUDA_VISIBLE_DEVICES / nvidia-smi."
        )
        GPU_ID = 0
    else:
        GPU_ID = _DESIRED_GPU_INDEX
else:
    GPU_ID = 0

from diffusers import DDIMScheduler

from launch_eval_runs import do_eval

ModuleNotFoundError: No module named 'launch_eval_runs'

In [13]:
# Set args
"""
model_choices:

stable-diffusion-xl
stable-diffusion-v1-5
stable-diffusion-v1-4
stable-diffusion-2-1
"""

args = dict(
    seed=0,
    output_dir="output", 
    eta=1.0,
    metrics_to_compute="ImageReward", 
    prompt_path='./prompt_files/image_rewards_benchmark.json', 
    model_name="stable-diffusion-xl", 
  )

fkd_args = dict(
    lmbda=2.0,
    num_particles=4,
    adaptive_resampling=True,
    resample_frequency=20,
    time_steps=100,
    potential_type='max',
    resampling_t_start=20,
    resampling_t_end=50,
    guidance_reward_fn='ImageReward',
    use_smc=False,
   )

args = argparse.Namespace(**args, **fkd_args)
args

Namespace(seed=0, output_dir='output', eta=1.0, metrics_to_compute='ImageReward', prompt_path='./prompt_files/image_rewards_benchmark.json', model_name='stable-diffusion-xl', lmbda=2.0, num_particles=4, adaptive_resampling=True, resample_frequency=20, time_steps=100, potential_type='max', resampling_t_start=20, resampling_t_end=50, guidance_reward_fn='ImageReward', use_smc=False)

In [14]:
args.num_inference_steps = fkd_args["time_steps"]
fkd_args

{'lmbda': 2.0,
 'num_particles': 4,
 'adaptive_resampling': True,
 'resample_frequency': 20,
 'time_steps': 100,
 'potential_type': 'max',
 'resampling_t_start': 20,
 'resampling_t_end': 50,
 'guidance_reward_fn': 'ImageReward',
 'use_smc': False}

In [15]:
# seed everything (GPU_ID is resolved in the first cell after import torch)
if torch.cuda.is_available():
    torch.manual_seed(args.seed)
    torch.cuda.manual_seed_all(args.seed)
    with torch.cuda.device(GPU_ID):
        torch.cuda.manual_seed(args.seed)

In [16]:
device = f"cuda:{GPU_ID}" if torch.cuda.is_available() else "cpu"
if torch.cuda.is_available():
    torch.cuda.set_device(GPU_ID)  # default "cuda" in rm_load uses this GPU

In [17]:
from fks_utils import get_model

pipeline = get_model(args.model_name)
pipeline = pipeline.to(device)    

Loading pipeline components...: 100%|██████████| 7/7 [00:01<00:00,  3.80it/s]


In [18]:
# set output directory
cur_time = datetime.now().strftime("%Y%m%d-%H%M%S")
output_dir = os.path.join(args.output_dir, cur_time)
os.makedirs(output_dir, exist_ok=False)
arg_path = os.path.join(output_dir, "args.json")
with open(arg_path, "w") as f:
    json.dump(vars(args), f, indent=4)

score_path = os.path.join(output_dir, "scores.jsonl")
images_path = os.path.join(output_dir, "images")
os.makedirs(images_path, exist_ok=False)

metrics_to_compute = args.metrics_to_compute.split("#")


# cache metric fns
do_eval(
    prompt=["test"],
    images=[Image.new("RGB", (224, 224))],
    metrics_to_compute=metrics_to_compute,
    )


load checkpoint from /home/adalal542/.cache/ImageReward/ImageReward.pt
checkpoint loaded


{'ImageReward': {'result': [-1.5143808126449585],
  'mean': -1.5143808126449585,
  'std': nan,
  'max': -1.5143808126449585,
  'min': -1.5143808126449585}}

In [19]:
# add prompts for generation
prompt_data = [
    {"prompt": "a photo of a brown knife and a blue donut"},
    {"prompt": "a photo of a blue clock and a white cup"},
    {"prompt": "a photo of an orange cow and a purple sandwich"},
    {"prompt": "a photo of a yellow bird and a black motorcycle"},
    {"prompt": "a photo of a green tennis racket and a black dog"},
    {"prompt": "a green stop sign in a red field"},    
]
len(prompt_data)

6

In [20]:
show_best_particle = True

In [23]:
with open(score_path, "w") as score_f:
    for prompt_idx, item in enumerate(prompt_data):
        if torch.cuda.is_available():
            torch.manual_seed(0)
            torch.cuda.manual_seed_all(0)
            with torch.cuda.device(GPU_ID):
                torch.cuda.manual_seed(0)
        
        
        prompt = [item['prompt']]*fkd_args['num_particles']
        start_time = datetime.now()
        
        images = pipeline(prompt, 
                          num_inference_steps=fkd_args["time_steps"], 
                          eta=args.eta,
                          fkd_args=fkd_args)
        end_time = datetime.now()        
        images = images[0]

        time_taken = end_time - start_time
        
        results = do_eval(prompt=prompt, images=images, metrics_to_compute=metrics_to_compute)
        guidance_reward = np.array(results["ImageReward"]["result"])
        sorted_idx = np.argsort(guidance_reward)[::-1]
        images = [images[i] for i in sorted_idx]
        
        results['time_taken'] = time_taken.total_seconds()
        results['prompt'] = prompt
        results['prompt_index'] = prompt_idx

        image_fpath = os.path.join(images_path, f"{prompt_idx}.png")
        results['image_path'] = image_fpath

        score_f.write(json.dumps(results) + "\n")
        
        if show_best_particle:
            _, ax = plt.subplots(1, 1, figsize=(5, 5))            
            ax.imshow(images[0])
            ax.axis("off")
        else:
            _, ax = plt.subplots(1, args.num_particles, figsize=(args.num_particles*5, 5))
            for i, image in enumerate(images):
                ax[i].imshow(image)
                ax[i].axis("off")
                
        plt.suptitle(prompt[0])
        plt.savefig(image_fpath)
        plt.show()
        plt.close()


Args: {'lmbda': 2.0, 'num_particles': 4, 'adaptive_resampling': True, 'resample_frequency': 20, 'time_steps': 100, 'potential_type': 'max', 'resampling_t_start': 20, 'resampling_t_end': 50, 'guidance_reward_fn': 'ImageReward', 'use_smc': False}


  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:16<00:00,  5.97it/s]


ModuleNotFoundError: No module named 'matplotlib_inline'